In [5]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np

In [6]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [7]:
resNet101_model = keras.applications.ResNet101(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling=None,
)

model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        resNet101_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.4),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet101 (Functional)      (None, None, None, 2048   42658176  
                             )                                   
                                                                 
 flatten_1 (Flatten)         (None, 2048)              0         
                                                                 
 dropout_2 (Dropout)         (None, 2048)              0         
                                                                 
 dense_3 (Dense)             (None, 512)               1049088   
                                                                 
 batch_normalization_2 (Bat  (None, 512)               2048      
 chNormalization)                                                
                                                                 
 activation_2 (Activation)   (None, 512)              

In [8]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Epoch 1/5
282/282 [==============================] - 574s 2s/step - loss: 2.6237 - accuracy: 0.2153 - val_loss: 3.1928 - val_accuracy: 0.0650
Epoch 2/5
282/282 [==============================] - 512s 2s/step - loss: 1.8377 - accuracy: 0.4464 - val_loss: 3.3016 - val_accuracy: 0.0860
Epoch 3/5
282/282 [==============================] - 512s 2s/step - loss: 1.4917 - accuracy: 0.5633 - val_loss: 2.3289 - val_accuracy: 0.3300
Epoch 4/5
282/282 [==============================] - 551s 2s/step - loss: 1.3199 - accuracy: 0.6112 - val_loss: 2.2607 - val_accuracy: 0.4180
Epoch 5/5
282/282 [==============================] - 574s 2s/step - loss: 1.1355 - accuracy: 0.6700 - val_loss: 1.5170 - val_accuracy: 0.5760


In [9]:
model.evaluate(x_test, testY)

63/63 [==============================] - 9s 143ms/step - loss: 1.5933 - accuracy: 0.5780


[1.5933341979980469, 0.578000009059906]